# Runtime Pose Backend Audit

Ce notebook reprend le script `runtime_pose_backend_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Compare les backends pose pour la latence de streaming.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Compare PyTorch and ONNX pose backends for final runtime candidates.
- Commande de reproduction referencee : pose backend runtime audit.
- Artefacts controles : Pose backend runtime audit exists. (`runs/exp_064_runtime_pose_backend_audit/runtime_pose_backend_audit.csv`).
- Run par defaut : `runs/exp_064_runtime_pose_backend_audit`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "runtime_pose_backend_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, write_json


VARIANT_SCORE_COLS = {
    "sequence_only_seed111_tcn_focal": "final_sequence_only_mean1",
    "fast_rule_seed111_small_mobilenet_crop_stride30": "final_attention_ppe_prior_mean1",
}


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


## Fonction `read_csv`

Cette cellule definit `read_csv`. Elle prepare une partie du script.

In [ ]:
def read_csv(path):
    path = resolve(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


## Fonction `metric_row`

Cette cellule definit `metric_row`. Elle prepare une partie du script.

In [ ]:
def metric_row(latency, variant, component):
    rows = latency[(latency["variant"].eq(variant)) & (latency["component"].eq(component))]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


## Fonction `first_alarm_time`

Cette cellule definit `first_alarm_time`. Elle prepare une partie du script.

In [ ]:
def first_alarm_time(df):
    alarms = df[df["alarm"].astype(int).eq(1)]
    if alarms.empty:
        return np.nan
    return float(alarms["time_s"].iloc[0])


## Fonction `compare_scores`

Cette cellule definit `compare_scores`. Elle prepare une partie du script.

In [ ]:
def compare_scores(reference_pred, candidate_pred, variant, score_col):
    ref = reference_pred[reference_pred["variant"].eq(variant)].copy()
    cand = candidate_pred[candidate_pred["variant"].eq(variant)].copy()
    if ref.empty or cand.empty or score_col not in ref.columns or score_col not in cand.columns:
        return {
            "score_rows_compared": 0,
            "score_max_abs_diff": np.nan,
            "score_mean_abs_diff": np.nan,
            "score_p95_abs_diff": np.nan,
            "alarm_diff_frames": np.nan,
            "reference_first_alarm_s": first_alarm_time(ref) if not ref.empty else np.nan,
            "candidate_first_alarm_s": first_alarm_time(cand) if not cand.empty else np.nan,
        }
    merged = ref[["frame", "alarm", score_col]].merge(
        cand[["frame", "alarm", score_col]],
        on="frame",
        suffixes=("_reference", "_candidate"),
    )
    if merged.empty:
        return {
            "score_rows_compared": 0,
            "score_max_abs_diff": np.nan,
            "score_mean_abs_diff": np.nan,
            "score_p95_abs_diff": np.nan,
            "alarm_diff_frames": np.nan,
            "reference_first_alarm_s": first_alarm_time(ref),
            "candidate_first_alarm_s": first_alarm_time(cand),
        }
    diff = (merged[f"{score_col}_reference"] - merged[f"{score_col}_candidate"]).abs().to_numpy(dtype=float)
    alarm_diff = int((merged["alarm_reference"].astype(int) != merged["alarm_candidate"].astype(int)).sum())
    return {
        "score_rows_compared": int(len(merged)),
        "score_max_abs_diff": float(np.max(diff)),
        "score_mean_abs_diff": float(np.mean(diff)),
        "score_p95_abs_diff": float(np.percentile(diff, 95)),
        "alarm_diff_frames": alarm_diff,
        "reference_first_alarm_s": first_alarm_time(ref),
        "candidate_first_alarm_s": first_alarm_time(cand),
    }


## Fonction `pass_bool`

Cette cellule definit `pass_bool`. Elle prepare une partie du script.

In [ ]:
def pass_bool(value):
    return bool(value) if pd.notna(value) else False


## Fonction `fmt`

Cette cellule definit `fmt`. Elle prepare une partie du script.

In [ ]:
def fmt(value, digits=3):
    if value is None or pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    out_dir = resolve(args.run_name if args.run_name.startswith("runs/") else f"runs/{args.run_name}")
    out_dir.mkdir(parents=True, exist_ok=True)

    reference_run = resolve(args.reference_run)
    candidate_run = resolve(args.candidate_run)
    reference_latency = read_csv(reference_run / "runtime_variant_latency_summary.csv")
    candidate_latency = read_csv(candidate_run / "runtime_variant_latency_summary.csv")
    reference_pred = read_csv(reference_run / "runtime_variant_predictions.csv")
    candidate_pred = read_csv(candidate_run / "runtime_variant_predictions.csv")

    rows = []
    for variant, score_col in VARIANT_SCORE_COLS.items():
        ref_pose = metric_row(reference_latency, variant, "pose")
        ref_total = metric_row(reference_latency, variant, "estimated_total")
        ref_processing = metric_row(reference_latency, variant, "variant_processing")
        cand_pose = metric_row(candidate_latency, variant, "pose")
        cand_total = metric_row(candidate_latency, variant, "estimated_total")
        cand_processing = metric_row(candidate_latency, variant, "variant_processing")
        if not all([ref_pose, ref_total, ref_processing, cand_pose, cand_total, cand_processing]):
            continue

        estimated_hybrid_mean = float(cand_pose["steady_mean_ms"]) + float(ref_processing["steady_mean_ms"])
        estimated_hybrid_p95 = float(cand_pose["steady_p95_ms"]) + float(ref_processing["steady_p95_ms"])
        score = compare_scores(reference_pred, candidate_pred, variant, score_col)
        rows.append(
            {
                "variant": variant,
                "score_col": score_col,
                "reference_backend": "pt_640_default",
                "candidate_backend": "onnx_320_cpu_pose",
                "reference_total_steady_mean_ms": float(ref_total["steady_mean_ms"]),
                "reference_total_steady_p95_ms": float(ref_total["steady_p95_ms"]),
                "reference_total_fps": float(ref_total["estimated_fps_from_steady_mean"]),
                "reference_pose_steady_mean_ms": float(ref_pose["steady_mean_ms"]),
                "reference_pose_steady_p95_ms": float(ref_pose["steady_p95_ms"]),
                "reference_processing_steady_mean_ms": float(ref_processing["steady_mean_ms"]),
                "reference_processing_steady_p95_ms": float(ref_processing["steady_p95_ms"]),
                "candidate_cpu_total_steady_mean_ms": float(cand_total["steady_mean_ms"]),
                "candidate_cpu_total_steady_p95_ms": float(cand_total["steady_p95_ms"]),
                "candidate_cpu_total_fps": float(cand_total["estimated_fps_from_steady_mean"]),
                "candidate_pose_steady_mean_ms": float(cand_pose["steady_mean_ms"]),
                "candidate_pose_steady_p95_ms": float(cand_pose["steady_p95_ms"]),
                "candidate_cpu_processing_steady_mean_ms": float(cand_processing["steady_mean_ms"]),
                "candidate_cpu_processing_steady_p95_ms": float(cand_processing["steady_p95_ms"]),
                "estimated_hybrid_cuda_score_total_mean_ms": estimated_hybrid_mean,
                "estimated_hybrid_cuda_score_total_p95_ms": estimated_hybrid_p95,
                "estimated_hybrid_cuda_score_fps": float(1000.0 / max(1e-9, estimated_hybrid_mean)),
                "pass_mean_30fps": estimated_hybrid_mean <= 33.34,
                "pass_p95_30fps": estimated_hybrid_p95 <= 33.34,
                **score,
            }
        )

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "runtime_pose_backend_audit.csv", index=False)
    write_json(
        out_dir / "runtime_pose_backend_audit_config.json",
        {
            "reference_run": str(reference_run),
            "candidate_run": str(candidate_run),
            "interpretation": "Candidate uses ONNX pose at fixed imgsz=320 on CPU. Hybrid total estimates combine candidate pose latency with reference CUDA scoring overhead because Ultralytics ONNX CPU disables CUDA visibility inside the same process on this machine.",
            "mean_30fps_budget_ms": 33.34,
            "p95_30fps_budget_ms": 33.34,
        },
    )

    lines = ["# Runtime Pose Backend Audit", ""]
    lines.append(
        "This audit tests whether the existing `yolo11n-pose.onnx` artifact improves the final fused runtime path. The ONNX file is fixed at `imgsz=320` and runs through ONNX Runtime CPU on this machine because the CUDA provider is missing required CUDA/cuBLAS dependencies."
    )
    lines.append("")
    lines.append("The reported hybrid estimate combines ONNX-CPU pose latency with the already-measured CUDA scoring overhead from the PyTorch 640 benchmark. This is an optimistic deployment estimate; the same-process ONNX-CPU run is also reported separately.")
    lines.append("")
    lines.append("| variant | ref total ms | ref FPS | ONNX pose ms | ONNX pose p95 | CPU total ms | hybrid total ms | hybrid p95 | hybrid FPS | score p95 diff | alarm diff | pass mean | pass p95 |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, row in df.iterrows():
        lines.append(
            f"| {row['variant']} | {fmt(row['reference_total_steady_mean_ms'])} | {fmt(row['reference_total_fps'])} | "
            f"{fmt(row['candidate_pose_steady_mean_ms'])} | {fmt(row['candidate_pose_steady_p95_ms'])} | "
            f"{fmt(row['candidate_cpu_total_steady_mean_ms'])} | {fmt(row['estimated_hybrid_cuda_score_total_mean_ms'])} | "
            f"{fmt(row['estimated_hybrid_cuda_score_total_p95_ms'])} | {fmt(row['estimated_hybrid_cuda_score_fps'])} | "
            f"{fmt(row['score_p95_abs_diff'])} | {int(row['alarm_diff_frames']) if pd.notna(row['alarm_diff_frames']) else 'NA'} | "
            f"{pass_bool(row['pass_mean_30fps'])} | {pass_bool(row['pass_p95_30fps'])} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- ONNX-CPU pose at 320 does not close the final fused real-time gap on this machine.")
    lines.append("- The optimistic hybrid estimate still fails both the 30 FPS mean budget and the p95 frame budget for the tested final fused candidate.")
    lines.append("- The ONNX CUDA provider is not currently usable because required CUDA/cuBLAS runtime DLLs are missing; fixing that dependency stack remains a separate deployment-engineering hypothesis.")
    (out_dir / "runtime_pose_backend_audit_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Compare PyTorch and ONNX pose backends for final runtime candidates.")
    parser.add_argument("--reference-run", default="runs/exp_053_realtime_variant_benchmark")
    parser.add_argument("--candidate-run", default="runs/exp_063_pose_backend_runtime_audit_onnx320_cpu")
    parser.add_argument("--run-name", default="exp_064_runtime_pose_backend_audit")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_064_runtime_pose_backend_audit_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["runtime_pose_backend_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
